# Ayar gold fund — NAV premium

Primary measure: unadjusted TSETMC closing price / same-date raw redemption NAV minus one. Positive is premium; negative is discount. NAV history is fund-published (Mofid), not TSETMC historical NAV. Prices and NAV are IRR per unit. No carry-forward. Refresh `commodity/gold/refresh.py --collect` first.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'commodity/gold').exists() and (p/'shared').exists())
if str(root) not in sys.path: sys.path.insert(0,str(root))
project = root/'commodity/gold'
bubble = pd.read_csv(project/'data/processed/bubble/ayar_nav_bubble.csv',parse_dates=['date'])
ranks = pd.read_csv(project/'data/processed/bubble/ayar_bubble_distribution.csv',parse_dates=['observation_date'])
display(bubble[['date','closing_price_irr','redemption_nav_irr','bubble_pct']].tail(10))


## Price premium to redemption NAV

Historical ex-post comparison. NAV date does not establish when that NAV became available to traders.


In [ ]:
fig = go.Figure(go.Scatter(x=bubble.date,y=bubble.bubble_pct,name='Close / redemption NAV',line_color='#B8860B'))
fig.add_hline(y=0,line_dash='dash')
fig.update_layout(title='Ayar: closing-price premium / discount to NAV',template='plotly_white',xaxis_title='Date',yaxis_title='Bubble (%)',hovermode='x unified')
fig.show()


## Signed distribution and historical ranks

Raw signed bubbles are not smoothed. Equal-weight rank uses all observations through that date; recent weighting uses exponential decay with a configurable 90-calendar-day half-life. Both include the current point and ties (<=); the first rank is 100. The histogram uses the entire loaded sample. Ranks are not reversal probabilities; publication-time and revision limitations prevent a vintage-safe trading backtest.


In [ ]:
from shared.market_analysis.bubble_percentiles import figures
for fig in figures(ranks,'Ayar: close vs redemption NAV'): fig.show()
display(ranks[['observation_date','bubble_pct','expanding_percentile','recent_weighted_percentile','history_count','effective_weighted_count']].tail(10))
display(bubble['bubble_pct'].describe(percentiles=[.1,.25,.5,.75,.9]).to_frame('Bubble (%)'))
